# Milestone 2 lab, notebook 5 of 8: Across two tables

## Where we are

| Notebook | What you do |
|---|---|
| 1. Empty in which rows | Write a check for empty cells that does not fire on good rows |
| 2. Unique on which columns | Find the receipt and the file that were counted twice |
| 3. A believable value | Catch 1,180 kg of bananas without catching 400 real bottles |
| 4. Rules of the business | Three rules a person must tell you. Each needs a join first |
| **5. Across two tables** (you are here) | Hold the receipts against the tills' own totals. Find the lines a join dropped |
| 6. A normal day | Find the store that sent nothing. Work out what a normal day is |
| 7. The false alarm | A holiday is not an error. See what silence costs |
| 8. The production version | The same checks as a tested program, and what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import keep, money, prepare_week, production, sql
from lotus_pipeline import inspect

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 24,384 sales lines.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: A check for the fault you did not think of

Look back at the last four notebooks. Each check caught one kind of fault, and you had to **imagine the fault first**: an empty amount, a receipt twice, a weight in grams, a refund with the wrong sign.

What about the fault nobody imagines?

In milestone 1, one thing proved the pipeline wrong without knowing why: the tills. Each till adds up its own day when it is closed. The store sends those totals in a second file, `till_totals`. That total never passes through the program that writes the sales lines.

Here it is for store `S03`, on the Monday of the refund.

In [2]:
sql("""
    SELECT business_date, store_id, till_id, receipts, gross_sales_cents, refunds_cents, net_total_cents
    FROM clean.till_totals
    WHERE store_id = 'S03' AND business_date = DATE '2026-08-24'
    ORDER BY till_id
""")

,business_date,store_id,till_id,receipts,gross_sales_cents,refunds_cents,net_total_cents
0,2026-08-24,S03,T1,18,42967,1008,41959
1,2026-08-24,S03,T2,16,35761,0,35761


One row per till per day. `net_total_cents` is the money the till took, less the money it gave back.

## Step 2: Prepare, then compare

The two tables do not have the same shape. `sales_lines` has one row per item line. `till_totals` has one row per till per day. So the lines must be **added up** to the till's shape first.

In [3]:
sql("""
    SELECT store_id, till_id, sales_date, count(*) AS lines, sum(amount_cents) AS lines_cents
    FROM clean.sales_lines
    WHERE store_id = 'S03' AND sales_date = DATE '2026-08-24'
    GROUP BY store_id, till_id, sales_date
    ORDER BY till_id
""")

,store_id,till_id,sales_date,lines,lines_cents
0,S03,T1,2026-08-24,55,43857
1,S03,T2,2026-08-24,58,35761


Till `T2`: 35761 in both tables. Till `T1`: the till says 41959, the lines say 43857. They are 1898 apart: $18.98.

Now do it for every till on every day, and keep only the rows where the two counts differ.

In [4]:
TILL_SIDE = """
    SELECT t.store_id, t.till_id, t.business_date AS sales_date,
           t.net_total_cents AS till_cents, coalesce(l.lines_cents, 0) AS lines_cents,
           coalesce(l.lines_cents, 0) - t.net_total_cents AS difference_cents
    FROM clean.till_totals t
    LEFT JOIN (
        SELECT store_id, till_id, sales_date, sum(amount_cents) AS lines_cents
        FROM clean.sales_lines
        GROUP BY store_id, till_id, sales_date
    ) l
      ON l.store_id = t.store_id AND l.till_id = t.till_id AND l.sales_date = t.business_date
    WHERE coalesce(l.lines_cents, 0) <> t.net_total_cents
"""
apart = sql(TILL_SIDE + " ORDER BY sales_date, store_id, till_id")
apart

,store_id,till_id,sales_date,till_cents,lines_cents,difference_cents
0,S03,T1,2026-08-24,41959,43857,1898
1,S01,T2,2026-08-25,37303,39110,1807
2,S04,T1,2026-08-25,31289,0,-31289
3,S04,T2,2026-08-25,27453,0,-27453
4,S05,T1,2026-08-27,25234,177302,152068
5,S02,T1,2026-08-28,66024,132048,66024
6,S02,T2,2026-08-28,37404,74808,37404
7,S06,T1,2026-08-29,55098,53900,-1198
8,S06,T2,2026-08-29,44525,43926,-599
9,S02,T2,2026-08-30,45657,43262,-2395


Ten rows. Read them one by one. You have met most of these stores already.

| Rows | Difference | What it is | You found it in |
|---|---|---|---|
| `S03`, Mon 24 | +1898 | The refund with the wrong sign | Notebook 4 |
| `S01`, Tue 25 | +1807 | One receipt written twice | Notebook 2 |
| `S04`, Tue 25, both tills | The lines are 0 | The tills counted money, and **no sales lines arrived at all** | Not yet |
| `S05`, Thu 27 | +152068 | The bananas in grams | Notebook 3 |
| `S02`, Fri 28, both tills | The lines are twice the till | The whole file twice | Notebook 2 |
| `S06`, Sat 29, both tills | −1198 and −599 | Lines are missing | Not yet |
| `S02`, Sun 30 | −2395 | The receipt stamped on the wrong day | Notebook 4 |

One query. It did not need to know about refunds, weights, files or clocks. It only asked: **do two counts of the same money agree?**

## Step 3: The other direction

That query started from the tills. Lines that have **no till total** to compare with would never show up in it. Start from the lines too.

In [5]:
LINES_SIDE = """
    SELECT l.store_id, l.till_id, l.sales_date, NULL AS till_cents,
           sum(l.amount_cents) AS lines_cents, sum(l.amount_cents) AS difference_cents
    FROM clean.sales_lines l
    LEFT JOIN clean.till_totals t
      ON t.store_id = l.store_id AND t.till_id = l.till_id AND t.business_date = l.sales_date
    WHERE t.store_id IS NULL
    GROUP BY l.store_id, l.till_id, l.sales_date
"""
sql(LINES_SIDE)

,store_id,till_id,sales_date,till_cents,lines_cents,difference_cents
0,S02,T2,2026-08-31,None,2395,2395


One row: $23.95 at `S02` on Monday 31 August, a day no till has closed yet. It is the other half of the receipt with the wrong clock. Put the two directions together, and you have the whole check.

In [6]:
yours = sql(TILL_SIDE + " UNION ALL " + LINES_SIDE)
keep("till_total", yours)

Your query:           11 bad rows
The production check: 11 bad rows  (The receipts add up to the till's own total)
The same rows.


Eleven rows, the same as the production check.

Comparing your numbers with another source that saw the same thing is called **reconciliation**. The till's own total has a name too: a **control total**. It is a total counted a second way, kept only to be compared. Nobody reads `till_totals` in a report. That is exactly its job.

## Step 4: What it cannot find

Reconciliation looks like the one check that sees everything. It does not. Look for Ines's butter: store `S04`, Friday 28 August.

In [7]:
butter = yours[(yours["store_id"] == "S04") & (yours["sales_date"].astype(str) == "2026-08-28")]
print(len(butter), "rows for S04 on 2026-08-28")

0 rows for S04 on 2026-08-28


Nothing. The till charged the old price itself. So the till's total and the lines are wrong **in the same way**, and they agree.

A control total only catches what goes wrong **after** the point where it was counted. A fault that is born in the till is in both counts.

| Fault | Born in | Does the till total see it? |
|---|---|---|
| Refund sign, receipt twice, grams, file twice | The program that writes the file | Yes |
| No file | The back office | Yes |
| Wrong clock | The till, but only in the time | Yes: the money is on the wrong day |
| Old butter price | The till, in the money itself | **No** |

That is why you still need the business rules of notebook 4.

## Step 5: The lines that are missing at `S06`

One row in the table is still unexplained. **Aiko Tanaka** manages store `S06`. On Saturday 29 August her two tills say she sold $17.97 more than her lines add up to.

Lines are missing. But from where? Count the rows before and after the cleaning step.

In [8]:
sql("""
    SELECT (SELECT count(*) FROM raw.sales_lines)   AS raw_rows,
           (SELECT count(*) FROM clean.sales_lines) AS clean_rows
""")

,raw_rows,clean_rows
0,24387,24384


Three rows went in and did not come out. In milestone 1 those two numbers were equal.

The cleaning step **joins** every sales line to its product. A line that finds no product is dropped. No error, no warning. Find the lines that have no partner. You must look in the **raw** table: the clean table has no row left to look at.

In [9]:
NO_PRODUCT = """
    SELECT l.line_id, l.store_id, CAST(CAST(l.sold_at AS TIMESTAMP) AS DATE) AS sales_date,
           l.product_id, CAST(l.amount_cents AS BIGINT) AS lost_cents
    FROM raw.sales_lines l
    LEFT JOIN clean.products p
      ON p.product_id = l.product_id
     AND CAST(CAST(l.sold_at AS TIMESTAMP) AS DATE)
         BETWEEN p.valid_from AND coalesce(p.valid_to, DATE '9999-12-31')
    LEFT JOIN clean.stores s ON s.store_id = l.store_id
    WHERE p.product_id IS NULL OR s.store_id IS NULL
"""
yours = sql(NO_PRODUCT)
print("Sales that the cleaning step dropped:", money(yours["lost_cents"].sum()))
yours

Sales that the cleaning step dropped: $17.97


,line_id,store_id,sales_date,product_id,lost_cents
0,S06-0829-000031,S06,2026-08-29,P-041,599
1,S06-0829-000091,S06,2026-08-29,P-041,599
2,S06-0829-000122,S06,2026-08-29,P-041,599


In [10]:
sql("SELECT * FROM clean.products WHERE product_id = 'P-041'")

,product_id,product_name,category,unit,unit_price_cents,valid_from,valid_to,status


Three lines of product `P-041`, at 599 each: $17.97. And `products` has no such product.

It is pumpkin soup, new this week. The tills already sell it. The buying team has not typed it into the price list yet.

In [11]:
keep("every_sale_has_a_product", yours)

Your query:           3 bad rows
The production check: 3 bad rows  (Every sale is for a product that exists)
The same rows.


A row that points to something that does not exist is called an **orphan row**. The rule it breaks is **referential integrity**: every row that points to another table finds a row there.

Notice how the two checks worked together. The till total said *"`S06` is $17.97 short"*. It could not say why. This check says why: three orphan rows.

## Step 6: Can you trust the control?

Every money check now leans on `till_totals`. If that table were wrong, the check would accuse good data, or miss bad data. A control you cannot trust is worse than none.

So the production code gives the control table three small checks of its own.

In [12]:
for key in ("one_total_per_till_per_day", "till_adds_up", "every_till_reported"):
    check = inspect.BY_KEY[key]
    print(f"{check.number}. {check.name:<46} {len(production(key))} bad rows")

14. A till has one total for a day                 0 bad rows


15. A till's total adds up                         0 bad rows
16. Every till of an open store sent its total     0 bad rows


All three pass. One total per till per day, each total adds up, and no till is missing.

## What you now know

- Reconciliation is comparing your numbers with another source that saw the same thing.
- A control total is a total counted a second way, kept only to be compared.
- Reconciliation finds faults you did not imagine. It says **that** something is wrong, not **what**.
- It cannot see a fault that is in both counts.
- Referential integrity: every row that points to another table finds a row there. An orphan row does not.
- A join can drop rows silently. Look for them in the table **before** the join.

**Say it in an interview:** "The most useful check was a reconciliation: sales lines added up per till per day, compared with the till's own closing total. One query found seven of our planted faults, including a store whose file never arrived. It could not find a wrong shelf price, because the till was wrong in the same way, so I kept the business rules beside it."

**The theory:** [SQL: Data Quality, Debugging & Auditing](https://www.d8loop.com/modules/knowledge-corner/sql/data-quality-debugging-auditing) for reconciliation and orphan rows. [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "What is good data?", the questions "Accurate?" and "Consistent?".

Next: [notebook 6](06_a_normal_day.ipynb). Store `S04` sent no file on Tuesday. Without the tills, would anything have noticed?